# Project 53 — Local SQL Analyst Agent

**Natural language to SQL to summary — query databases by asking questions in English.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Database | SQLite (local) |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to connect an LLM to a **live SQL database** for natural-language querying
2. How to generate **safe, read-only SQL** from English questions
3. How to **summarize query results** in plain English
4. How to validate generated SQL before execution
5. How to build a complete **NL → SQL → Summary** pipeline locally

## 2 · Why This Project Matters

Most business data lives in relational databases. Non-technical stakeholders can't
write SQL but need answers fast. A local NL-to-SQL agent lets anyone query data
safely while keeping sensitive data on-premise.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`
- Python packages below (SQLite is built into Python)

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community

## 4 · Imports and Configuration

In [ ]:
import sqlite3
import json
import os
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "qwen3:8b"
DB_PATH = "sample_data/company.db"

llm = ChatOllama(model=MODEL, temperature=0.1)
print(f"LLM ready: {MODEL}")

## 5 · Create Sample Database

We create a realistic company database with employees, departments, and sales tables.
In production this would be your existing database.

In [ ]:
import random
random.seed(42)

Path('sample_data').mkdir(exist_ok=True)

# Remove old DB if exists
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Create tables
cur.executescript('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    budget REAL NOT NULL
);

CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    salary REAL NOT NULL,
    hire_date TEXT NOT NULL,
    title TEXT NOT NULL,
    FOREIGN KEY (department_id) REFERENCES departments(id)
);

CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    employee_id INTEGER,
    amount REAL NOT NULL,
    sale_date TEXT NOT NULL,
    product TEXT NOT NULL,
    FOREIGN KEY (employee_id) REFERENCES employees(id)
);
''')

# Populate departments
depts = [('Engineering', 500000), ('Sales', 300000), ('Marketing', 200000),
         ('HR', 150000), ('Finance', 250000)]
cur.executemany('INSERT INTO departments (name, budget) VALUES (?, ?)', depts)

# Populate employees
first_names = ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace',
               'Henry', 'Iris', 'Jack', 'Karen', 'Leo', 'Mona', 'Nick', 'Olivia',
               'Paul', 'Quinn', 'Rita', 'Sam', 'Tina']
titles = ['Junior', 'Senior', 'Lead', 'Manager', 'Director']
for i, name in enumerate(first_names, 1):
    dept_id = random.randint(1, 5)
    salary = random.randint(50000, 150000)
    year = random.randint(2018, 2024)
    month = random.randint(1, 12)
    hire_date = f'{year}-{month:02d}-01'
    title = random.choice(titles)
    cur.execute('INSERT INTO employees (name, department_id, salary, hire_date, title) VALUES (?, ?, ?, ?, ?)',
               (name, dept_id, salary, hire_date, title))

# Populate sales
products = ['Widget A', 'Widget B', 'Premium C', 'Enterprise D']
for i in range(100):
    emp_id = random.randint(1, 20)
    amount = round(random.uniform(500, 50000), 2)
    year = random.choice([2023, 2024])
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    sale_date = f'{year}-{month:02d}-{day:02d}'
    product = random.choice(products)
    cur.execute('INSERT INTO sales (employee_id, amount, sale_date, product) VALUES (?, ?, ?, ?)',
               (emp_id, amount, sale_date, product))

conn.commit()

# Verify
for table in ['departments', 'employees', 'sales']:
    count = cur.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count} rows')

conn.close()
print(f'\nDatabase created at {DB_PATH}')

## 6 · Database Schema Inspector

The LLM needs to know the database schema to generate correct SQL. We extract it
automatically.

In [ ]:
def get_schema(db_path: str) -> str:
    """Extract the full schema from a SQLite database."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL")
    schemas = [row[0] for row in cur.fetchall()]
    
    # Also get sample data for each table
    tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    samples = []
    for (table_name,) in tables:
        rows = cur.execute(f'SELECT * FROM {table_name} LIMIT 3').fetchall()
        cols = [d[0] for d in cur.description]
        samples.append(f'-- {table_name} sample: columns={cols}, rows={rows}')
    
    conn.close()
    return '\n\n'.join(schemas) + '\n\n' + '\n'.join(samples)


schema = get_schema(DB_PATH)
print(schema)

## 7 · Build the NL-to-SQL Pipeline

The pipeline has three stages:
1. **Generate SQL** from the question + schema
2. **Execute SQL** on the database (read-only)
3. **Summarize** the results in plain English

In [ ]:
def nl_to_sql(question: str, db_path: str = DB_PATH, verbose: bool = True) -> dict:
    """Convert natural language to SQL, execute, and summarize."""
    schema_text = get_schema(db_path)

    # Step 1: Generate SQL
    sql_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a SQL expert. Given this database schema:\n"
         "{schema}\n\n"
         "Generate a SQLite query to answer the question.\n"
         "Rules:\n"
         "- Return ONLY the SQL query, no explanations\n"
         "- Use only SELECT statements (read-only)\n"
         "- Do not use DELETE, UPDATE, INSERT, DROP, or ALTER\n"
         "- Use proper JOINs when needed\n"
         "- Alias columns for readability"),
        ("human", "{question}"),
    ])
    sql_chain = sql_prompt | llm | StrOutputParser()
    raw_sql = sql_chain.invoke({"schema": schema_text, "question": question})

    # Clean SQL
    sql_lines = []
    for line in raw_sql.strip().split("\n"):
        line = line.strip()
        if line and not line.startswith("```"):
            sql_lines.append(line)
    clean_sql = " ".join(sql_lines)

    # Safety check
    forbidden = ['DELETE', 'UPDATE', 'INSERT', 'DROP', 'ALTER', 'CREATE']
    sql_upper = clean_sql.upper()
    for word in forbidden:
        if word in sql_upper:
            return {"question": question, "sql": clean_sql,
                    "result": "BLOCKED: Unsafe SQL detected", "summary": "Query blocked for safety"}

    # Step 2: Execute SQL
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute(clean_sql)
        columns = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall()
        conn.close()
        result_text = f"Columns: {columns}\nRows ({len(rows)}): {rows[:20]}"
    except Exception as e:
        result_text = f"SQL Error: {e}"
        columns, rows = [], []

    # Step 3: Summarize
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a data analyst. Summarize these SQL query results in 2-3 clear\n"
         "sentences for a business stakeholder. Be specific with numbers."),
        ("human",
         "Question: {question}\nSQL: {sql}\nResults: {results}"),
    ])
    summary_chain = summary_prompt | llm | StrOutputParser()
    summary = summary_chain.invoke({
        "question": question, "sql": clean_sql, "results": result_text})

    if verbose:
        print(f"\n{'='*60}")
        print(f"Q: {question}")
        print(f"{'='*60}")
        print(f"SQL: {clean_sql}")
        print(f"\nRaw result: {result_text[:300]}")
        print(f"\nSummary: {summary}")

    return {"question": question, "sql": clean_sql,
            "result": result_text, "summary": summary}


print("NL-to-SQL pipeline ready.")

## 8 · Run Queries

Let's test with questions of increasing complexity.

In [ ]:
r1 = nl_to_sql("How many employees are in each department?")

In [ ]:
r2 = nl_to_sql("Who are the top 5 highest-paid employees and which departments are they in?")

In [ ]:
r3 = nl_to_sql("What is the total sales amount by product in 2024?")

In [ ]:
r4 = nl_to_sql("Which department has the highest average salary?")

In [ ]:
r5 = nl_to_sql("Show me employees who made more than $10,000 in total sales")

## 9 · Validate Results

Let's verify a few results against known ground truth.

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Ground truth: employee count
gt_count = cur.execute('SELECT COUNT(*) FROM employees').fetchone()[0]
print(f'Total employees (ground truth): {gt_count}')

# Ground truth: total sales 2024
gt_sales = cur.execute("SELECT SUM(amount) FROM sales WHERE sale_date LIKE '2024%'").fetchone()[0]
print(f'Total 2024 sales (ground truth): ${gt_sales:,.2f}')

# Ground truth: dept with highest avg salary
gt_dept = cur.execute('''
    SELECT d.name, AVG(e.salary) as avg_sal
    FROM employees e JOIN departments d ON e.department_id = d.id
    GROUP BY d.name ORDER BY avg_sal DESC LIMIT 1
''').fetchone()
print(f'Highest avg salary dept (ground truth): {gt_dept[0]} (${gt_dept[1]:,.0f})')

conn.close()

# Check if LLM results are consistent
print('\nValidation: Check that LLM summaries align with ground truth above.')

## 10 · Save Results

In [ ]:
results = [r1, r2, r3, r4, r5]
with open("sql_analysis_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Saved {len(results)} query results")

## 11 · Limitations

- **SQL injection risk** — the safety check is basic; production needs parameterized queries
- **Complex JOINs** can trip up small models — larger models handle them better
- **No multi-turn context** — each question is independent
- **Schema size limits** — very large schemas may exceed context window
- Only tested with SQLite; other databases need dialect adjustments

## 12 · How to Improve

1. **Add query validation**: parse the SQL AST before execution
2. **Multi-turn conversations**: remember previous queries for follow-up questions
3. **Support PostgreSQL/MySQL**: add SQLAlchemy for database-agnostic queries
4. **Add visualization**: auto-generate charts from query results
5. **Few-shot examples**: include example Q&A pairs in the prompt for better accuracy

## 13 · Mini Challenge

1. Add a `projects` table and write a question that requires a 3-way JOIN
2. Implement a retry mechanism that feeds SQL errors back to the LLM
3. Add a `explain_sql` function that asks the LLM to explain the generated query

## 14 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| NL-to-SQL | LLM generates SQL from English + schema context |
| Safety | Read-only enforcement with forbidden keyword detection |
| Result summarization | LLM explains raw query results for stakeholders |
| Schema extraction | Automated schema + sample data extraction |
| Local-first | SQLite + Ollama — no cloud needed |